<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/registrasion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uzaysal Normalizasyon (MNI152 Registration)
Farklı bireylerden alınan MR görüntüleri; kafa yapısı, boyutu ve beyin oryantasyonu açısından doğal bir çeşitlilik gösterir. Bu çeşitlilik, makine öğrenimi modellerinin doku değişimlerini standart bir düzlemde analiz etmesini zorlaştırır. Uzaysal Normalizasyon (Registration) işlemi, tüm beyin görüntülerini dünya çapında standart kabul edilen MNI152 (Montreal Neurological Institute) şablonu üzerine hizalayarak verileri ortak bir koordinat sistemine taşır.

# Neden MNI Hizalaması Tercih Edildi?
Bireyler arasındaki kafa yapısı ve beyin oryantasyonu farklılıklarını gidermek amacıyla, tüm görüntüler dünya genelinde standart kabul edilen MNI152 koordinat sistemine taşınmıştır. Bu süreçte kullanılan ANTS SyN algoritması, beyin dokusundaki şekil bozukluklarını doğrusal olmayan bir şekilde düzelterek hipokampus gibi kritik bölgelerin tüm hastalarda aynı koordinatlara gelmesini sağlar. Skull stripping aşamasından gelen maskelerin sürece dahil edilmesiyle algoritmanın doğrudan beyin dokusuna odaklanması sağlanmış, böylece derin öğrenme modelinin Alzheimer kaynaklı doku kayıplarını en yüksek hassasiyetle ve standart bir veri yapısı üzerinden öğrenmesi garanti altına alınmıştır.

In [ ]:
!pip install antspyx -q

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("DİKKAT: GPU bulunamadı!")

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
CONFIG_CN = {
    "kategori"    : "CN",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_CN',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionn/Registrasion_CN'
}


In [ ]:
CONFIG_EMCI = {
    "kategori"    : "EMCI",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_EMCI',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionn/Registrasion_EMCI'
}


In [ ]:
import ants

mni_sablon = ants.get_ants_data('mni')
mni_img    = ants.image_read(mni_sablon)

print(f"MNI Şablonu : {mni_sablon}")
print(f"Boyut       : {mni_img.shape}")
print(f"Spacing     : {mni_img.spacing}")

In [ ]:
import os
import shutil
import ants

def batch_mni_hizalama(config, mni_sablon, batch_size=20):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    etiket = config["kategori"]

    local_in = "/content/batch_in_mni"
    local_out = "/content/batch_out_mni"

    print(f"\n {etiket} MNI Registration Başlıyor (Grup Boyutu: {batch_size})")

    # 1. Analiz Aşaması
    all_pending = []
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])

    toplam_dosya = 0
    atlanan_dosya = 0
    basarili_dosya = 0
    hatali_dosya = 0

    print(" Klasörler taranıyor, lütfen bekleyin...", end="\r")

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        seanslar = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]

        for seans in seanslar:
            seans_yolu = os.path.join(subject_path, seans)
            dosyalar = os.listdir(seans_yolu)
            beyin_dosyasi = next((f for f in dosyalar if f.endswith('.nii.gz') and '_bet' not in f), None)
            maske_dosyasi = next((f for f in dosyalar if f.endswith('_bet.nii.gz')), None)

            if beyin_dosyasi:
                toplam_dosya += 1
                rel_path = os.path.join(subject_id, seans)
                cikti_yolu = os.path.join(output_base, rel_path, beyin_dosyasi)

                if not os.path.exists(cikti_yolu):
                    all_pending.append((os.path.join(seans_yolu, beyin_dosyasi),
                                      os.path.join(seans_yolu, maske_dosyasi) if maske_dosyasi else None,
                                      rel_path, beyin_dosyasi, subject_id))
                else:
                    atlanan_dosya += 1

    # İşleme başlamadan önce özet yapısı veriliyor.
    kalan_dosya = len(all_pending)
    print(" "*50, end="\r")
    print(f" KLASÖR ANALİZİ ({etiket}):")
    print(f"    Toplam Bulunan Dosya : {toplam_dosya}")
    print(f"    Zaten İşlenmiş Olan  : {atlanan_dosya}")
    print(f"    Şimdi İşlenecek Olan : {kalan_dosya}")
    print("-" * 40)


    if not all_pending:
        print(f" İşlenecek yeni dosya yok. Sistem durduruluyor.")
    else:

        for i in range(0, kalan_dosya, batch_size):
            batch = all_pending[i:i+batch_size]
            shutil.rmtree(local_in, ignore_errors=True)
            shutil.rmtree(local_out, ignore_errors=True)
            os.makedirs(local_in, exist_ok=True)
            os.makedirs(local_out, exist_ok=True)

            batch_subjects = sorted(list(set([item[4] for item in batch])))
            print(f"\n Grup {i//batch_size + 1} Başlıyor...")
            print(f" İşlenen Hastalar: {', '.join(batch_subjects)}")

            mapping = {}
            for full_src, mask_src, rel_p, fname, sid in batch:
                unique_name = rel_p.replace("/", "_") + "___" + fname
                local_girdi = os.path.join(local_in, unique_name)
                shutil.copy2(full_src, local_girdi)

                local_maske = None
                if mask_src:
                    local_maske = os.path.join(local_in, unique_name.replace(".nii.gz", "_mask.nii.gz"))
                    shutil.copy2(mask_src, local_maske)

                mapping[unique_name] = (rel_p, fname, local_girdi, local_maske)

            print(f" MNI Hizalama  çalışıyor... ", end="", flush=True)

            try:
                for u_name, (rel_p, fname, l_girdi, l_maske) in mapping.items():
                    l_out_file = os.path.join(local_out, u_name)
                    img = ants.image_read(l_girdi)
                    if l_maske:
                        mask = ants.image_read(l_maske)
                        img = img * mask
                        reg = ants.registration(fixed=mni_sablon, moving=img, type_of_transform='antsRegistrationSyNQuick[a]', moving_mask=mask)
                    else:
                        reg = ants.registration(fixed=mni_sablon, moving=img, type_of_transform='antsRegistrationSyNQuick[a]')
                    ants.image_write(reg['warpedmovout'], l_out_file)

                print(" Bitti")

                print(f" Drive'a aktarılıyor... ", end="", flush=True)
                for u_name, (rel_p, fname, _, _) in mapping.items():
                    l_res = os.path.join(local_out, u_name)
                    d_target_dir = os.path.join(output_base, rel_p)
                    os.makedirs(d_target_dir, exist_ok=True)
                    if os.path.exists(l_res):
                        shutil.move(l_res, os.path.join(d_target_dir, fname))
                        basarili_dosya += 1


                su_an_biten = min(i+batch_size, kalan_dosya)
                print(f" {su_an_biten}/{kalan_dosya} yeni dosya kaydedildi.")

            except Exception as e:
                print(f" Grup Hatası: {str(e)}")
                hatali_dosya += len(batch)

    # Kodun sonunda özet bilgisi veriliyor.
    print(f"\n{etiket} MNI Registration Özeti:")
    print(f"Toplam   : {toplam_dosya}")
    print(f"Başarılı : {basarili_dosya}")
    print(f"Atlanan  : {atlanan_dosya}")
    print(f"Hatalı   : {hatali_dosya}")
    print(f"{etiket} grubu tamamlandı.")

In [ ]:
batch_mni_hizalama(CONFIG_CN, mni_img)

In [ ]:
batch_mni_hizalama(CONFIG_EMCI, mni_img)

In [ ]:
import os

def mni_kontrol(config):
    hedef  = config["hedef"]
    etiket = config["kategori"]

    goruntuler_sayisi = 0
    for root, dirs, files in os.walk(hedef):
        for file in files:
            if file.endswith('.nii.gz'):
                goruntuler_sayisi += 1

    print(f"{etiket} MNI Klasörü:")
    print(f"   Hizalanmış görüntü sayısı : {goruntuler_sayisi}")
    print(f"    Tüm işlenmiş görüntülerin kontrolü tamamlandı.")

mni_kontrol(CONFIG_CN)
print()
mni_kontrol(CONFIG_EMCI)